# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước.
Notebook này chứa mã nguồn hoàn chỉnh để chạy toàn bộ thí nghiệm (E0 - E5) trên **Kaggle GPU** (hoặc Google Colab T4).


## 0. Môi trường (Colab T4 / Kaggle GPU)
Kiểm tra phần cứng GPU và cài đặt thư viện cần thiết.
Hỗ trợ tự động nhận diện môi trường Kaggle (`/kaggle/working`) và Google Colab.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# Kiểm tra GPU
!nvidia-smi

# Nhận diện môi trường Kaggle / Colab
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle/working')
IN_COLAB = 'google.colab' in sys.modules

print(f"Môi trường phát hiện: {'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'Local'}")

# Nếu chạy trên Kaggle mà repo chưa có trong thư mục hiện tại:
if IN_KAGGLE:
    if not os.path.exists("scripts/prepare_data.py"):
        repo_dir = "/kaggle/working/K4-Track-4-Day10-VisionLanguageModel"
        if not os.path.exists(repo_dir):
            subprocess.run(["git", "clone", "https://github.com/chung170703/K4-Track-4-Day10-VisionLanguageModel.git", repo_dir], check=True)
        os.chdir(repo_dir)
        sys.path.insert(0, repo_dir)
    print(f"Thư mục làm việc hiện tại: {os.getcwd()}")

# Cài đặt thư viện theo requirements-colab.txt
!pip install -q -r requirements-colab.txt

# Thiết lập thông tin xác thực Kaggle (nếu cần tải dataset qua CLI)
if IN_COLAB:
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    except Exception:
        pass
elif IN_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        os.environ['KAGGLE_USERNAME'] = secrets.get_secret("KAGGLE_USERNAME")
        os.environ['KAGGLE_KEY'] = secrets.get_secret("KAGGLE_KEY")
    except Exception:
        pass


## 1. Dữ liệu: tải và chia split
Tải dataset Road Damage và chia split theo phiên quay (session) cố định để tránh rò rỉ dữ liệu.
Trên Kaggle, nếu bạn đã bấm `+ Add Input` -> dataset `lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes`, code sẽ tự động nhận diện từ `/kaggle/input`!


In [ ]:
# Tải và chuẩn bị dữ liệu
import os
import sys
import json
import subprocess
from pathlib import Path

kaggle_input_found = None
if os.path.exists("/kaggle/input"):
    for p in Path("/kaggle/input").rglob("*"):
        if p.is_dir() and (p / "images").is_dir() and (p / "labels-YOLO").is_dir():
            kaggle_input_found = str(p)
            break

if not os.path.exists("data/splits/meta.json"):
    if kaggle_input_found:
        print(f"Tìm thấy dataset từ Kaggle input: {kaggle_input_found}")
        subprocess.run([sys.executable, "scripts/prepare_data.py", "--source", kaggle_input_found], check=True)
    elif os.environ.get("KAGGLE_KEY") or os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
        print("Tải dataset qua Kaggle CLI...")
        subprocess.run([sys.executable, "scripts/prepare_data.py", "--kaggle"], check=True)
    elif os.path.exists("work/raw"):
        subprocess.run([sys.executable, "scripts/prepare_data.py", "--source", "work/raw"], check=True)
    else:
        raise RuntimeError("Không tìm thấy dữ liệu. Hãy thêm Kaggle Dataset 'lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes' vào notebook.")
else:
    print("Dữ liệu đã được chuẩn bị sẵn trong data/")

# Đọc metadata và in thông số phân bố
with open("data/splits/meta.json", "r", encoding="utf-8") as f:
    meta = json.load(f)
print(f"Dataset Version: {meta['dataset_version']}")
for s in ("train", "val", "test"):
    c = meta["counts"][s]
    print(f"  {s:5s}: {c['images']:4d} ảnh | boxes: {c['boxes']}")


In [ ]:
# Trực quan hoá 6 mẫu từ train.jsonl kèm bounding box
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_records = load_jsonl("data/splits/train.jsonl")
print(f"Số lượng mẫu huấn luyện: {len(train_records)}")

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
colors = {"pothole": "red", "crack": "orange", "manhole": "cyan"}

for i in range(min(6, len(train_records))):
    sample = train_records[i]
    img = Image.open(f"data/{sample['image_path']}")
    w, h = sample["width"], sample["height"]
    ax = axes[i]
    ax.imshow(img)
    ax.set_title(f"{sample['image_id']} (session: {sample['session']})", fontsize=10)
    ax.axis("off")
    for obj in sample["objects"]:
        xc, yc, bw, bh = obj["yolo"]
        x1 = (xc - bw / 2) * w
        y1 = (yc - bh / 2) * h
        box_w = bw * w
        box_h = bh * h
        c = colors.get(obj["label"], "yellow")
        rect = patches.Rectangle((x1, y1), box_w, box_h, linewidth=2, edgecolor=c, facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, max(0, y1 - 4), obj["label"], color="white", fontsize=8,
                bbox=dict(boxstyle="square,pad=0.1", fc=c, ec="none", alpha=0.8))

plt.tight_layout()
os.makedirs("report/images", exist_ok=True)
plt.savefig("report/images/train_samples_visualization.png", dpi=150)
plt.show()


## 2. Nhãn: YOLO → JSON thang 0–1000
Chuyển đổi nhãn YOLO toạ độ thực [0, 1] sang định dạng JSON toạ độ nguyên [0, 1000] theo quy ước grounding của Qwen3-VL.
Các box được sắp xếp theo thứ tự `(y1, x1)` (từ trên xuống dưới, từ trái sang phải) để tạo chuỗi sinh ổn định cho LLM.


In [ ]:
CLASSES = ("pothole", "crack", "manhole")

def yolo_to_box(label, yolo):
    """    Chuyển toạ độ YOLO [xc, yc, w, h] chuẩn hoá [0,1] sang [x1, y1, x2, y2]
    thang số nguyên [0, 1000].
    """
    xc, yc, w, h = yolo
    x1 = (xc - w / 2.0) * 1000.0
    y1 = (yc - h / 2.0) * 1000.0
    x2 = (xc + w / 2.0) * 1000.0
    y2 = (yc + h / 2.0) * 1000.0
    
    ix1 = max(0, min(1000, int(round(x1))))
    iy1 = max(0, min(1000, int(round(y1))))
    ix2 = max(0, min(1000, int(round(x2))))
    iy2 = max(0, min(1000, int(round(y2))))
    
    # Đảm bảo box hợp lệ x2 > x1 và y2 > y1
    if ix2 <= ix1:
        ix2 = min(1000, ix1 + 1)
        if ix2 <= ix1:
            ix1 = max(0, ix2 - 1)
    if iy2 <= iy1:
        iy2 = min(1000, iy1 + 1)
        if iy2 <= iy1:
            iy1 = max(0, iy2 - 1)
            
    return {"label": label, "bbox_2d": [ix1, iy1, ix2, iy2]}

def objects_to_answer(objects):
    """    Chuyển danh sách object sang chuỗi JSON list, sắp xếp theo (y1, x1).
    """
    boxes = [yolo_to_box(obj["label"], obj["yolo"]) for obj in objects]
    boxes.sort(key=lambda b: (b["bbox_2d"][1], b["bbox_2d"][0]))
    return json.dumps(boxes, separators=(",", ":"))

PROMPT = (
    "Locate road damage in this image. Detect objects belonging to these 3 classes: pothole, crack, manhole. "
    'Return ONLY a JSON list formatted as [{"label": "<class>", "bbox_2d": [x1, y1, x2, y2]}]. '
    "Coordinates must be integers on a 0-1000 scale relative to image width and height. "
    "If no road damage is detected, return []."
)

# Kiểm tra tính toàn vẹn 2 chiều
sample = train_records[0]
answer_str = objects_to_answer(sample["objects"])
print("Sample answer JSON string:")
print(answer_str)

# Kiểm tra độ lệch pixel khi chuyển ngược
for orig_obj in sample["objects"]:
    box_dict = yolo_to_box(orig_obj["label"], orig_obj["yolo"])
    oxc, oyc, ow, oh = orig_obj["yolo"]
    ox1, oy1 = (oxc - ow/2)*sample["width"], (oyc - oh/2)*sample["height"]
    px1, py1 = box_dict["bbox_2d"][0]/1000*sample["width"], box_dict["bbox_2d"][1]/1000*sample["height"]
    diff_x = abs(ox1 - px1)
    diff_y = abs(oy1 - py1)
    assert diff_x <= 1.0 and diff_y <= 1.0, f"Sai lệch toạ độ: diff_x={diff_x}, diff_y={diff_y}"
print("Kiểm tra chuyển đổi toạ độ: HOÀN TOÀN CHÍNH XÁC (sai lệch < 1 pixel).")


## 3. E0: zero-shot
Đánh giá mô hình gốc Qwen3-VL-4B-Instruct ở chế độ Zero-shot (không huấn luyện).
Mô hình được nạp ở định dạng 4-bit NF4, giữ ViT và `lm_head` ở fp16.


In [ ]:
import torch
import gc
from tqdm import tqdm
from transformers import BitsAndBytesConfig, AutoProcessor

try:
    from transformers import Qwen3VLForConditionalGeneration
    model_cls = Qwen3VLForConditionalGeneration
except ImportError:
    from transformers import AutoModelForImageTextToText
    model_cls = AutoModelForImageTextToText

model_id = "Qwen/Qwen3-VL-4B-Instruct"

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=["visual", "lm_head"]
)

print(f"Loading {model_id} in 4-bit NF4...")
processor = AutoProcessor.from_pretrained(model_id)
processor.tokenizer.padding_side = "left"

base_model = model_cls.from_pretrained(
    model_id,
    quantization_config=quantization,
    torch_dtype=torch.float16,
    device_map={"": 0}
)
base_model.eval()

def run_inference(model, processor, split="test", batch_size=4, out_file="work/e0.jsonl"):
    os.makedirs(os.path.dirname(out_file), exist_ok=True)
    records = load_jsonl(f"data/splits/{split}.jsonl")
    results = []
    
    with open(out_file, "w", encoding="utf-8") as f_out:
        for i in tqdm(range(0, len(records), batch_size), desc=f"Evaluating {split}"):
            batch = records[i:i + batch_size]
            batch_messages = []
            for entry in batch:
                img = Image.open(f"data/{entry['image_path']}").convert("RGB")
                batch_messages.append([
                    {
                        "role": "user",
                        "content": [
                            {"type": "image", "image": img},
                            {"type": "text", "text": PROMPT}
                        ]
                    }
                ])
            
            inputs = processor.apply_chat_template(
                batch_messages,
                tokenize=True,
                add_generation_prompt=True,
                return_dict=True,
                return_tensors="pt",
                padding=True
            ).to("cuda")
            
            prompt_len = inputs.input_ids.shape[1]
            with torch.no_grad():
                out = model.generate(
                    **inputs,
                    max_new_tokens=256,
                    do_sample=False
                )
            
            for j, entry in enumerate(batch):
                gen_ids = out[j, prompt_len:]
                raw = processor.tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
                row = {"image_id": entry["image_id"], "raw": raw}
                f_out.write(json.dumps(row, ensure_ascii=False) + "\n")
                results.append(row)
                
    print(f"Đã lưu {len(results)} kết quả dự đoán vào {out_file}")
    return results

# Thực hiện suy luận Zero-shot E0
if not os.path.exists("work/e0.jsonl"):
    run_inference(base_model, processor, split="test", batch_size=4, out_file="work/e0.jsonl")


In [ ]:
# Chấm điểm E0
import sys
import subprocess
subprocess.run([sys.executable, "scripts/score.py", "--pred", "work/e0.jsonl", "--name", "E0", "--out", "report/evidence/e0.json"], check=True)


## 4. Dựng dữ liệu huấn luyện: chat template và mask loss
Áp dụng chat template chuẩn và mask loss:
- Prompt và token hình ảnh được đặt `labels = -100` để mô hình chỉ học phần văn bản trả lời (JSON).
- Token hình ảnh kích thước 640×360 tương ứng với 220 token visual.


In [ ]:
from torch.utils.data import Dataset, DataLoader

class RoadDamageDataset(Dataset):
    def __init__(self, jsonl_path, processor, prompt=PROMPT):
        self.records = load_jsonl(jsonl_path)
        self.processor = processor
        self.prompt = prompt
        
    def __len__(self):
        return len(self.records)
        
    def __getitem__(self, idx):
        entry = self.records[idx]
        image = Image.open(f"data/{entry['image_path']}").convert("RGB")
        answer = objects_to_answer(entry["objects"])
        
        user_content = [
            {"type": "image", "image": image},
            {"type": "text", "text": self.prompt}
        ]
        user_msg = {"role": "user", "content": user_content}
        assistant_msg = {"role": "assistant", "content": [{"type": "text", "text": answer}]}
        
        prompt_enc = self.processor.apply_chat_template(
            [user_msg],
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt"
        )
        
        full_enc = self.processor.apply_chat_template(
            [user_msg, assistant_msg],
            add_generation_prompt=False,
            tokenize=True,
            return_dict=True,
            return_tensors="pt"
        )
        
        n = prompt_enc.input_ids.shape[1]
        assert torch.equal(full_enc.input_ids[0, :n], prompt_enc.input_ids[0]), "Prompt phải là tiền tố của full sequence!"
        
        labels = full_enc.input_ids.clone()
        labels[:, :n] = -100  # Mask toàn bộ prompt tokens
        
        item = {k: v.squeeze(0) for k, v in full_enc.items()}
        item["labels"] = labels.squeeze(0)
        return item

def collate_fn(batch, pad_token_id):
    max_len = max(ex["input_ids"].shape[0] for ex in batch)
    
    batch_input_ids = []
    batch_attn_mask = []
    batch_labels = []
    has_mm = "mm_token_type_ids" in batch[0]
    batch_mm = []
    
    for ex in batch:
        cur_len = ex["input_ids"].shape[0]
        pad_len = max_len - cur_len
        
        if pad_len > 0:
            pad_ids = torch.full((pad_len,), pad_token_id, dtype=ex["input_ids"].dtype)
            pad_mask = torch.zeros((pad_len,), dtype=ex["attention_mask"].dtype)
            pad_lbls = torch.full((pad_len,), -100, dtype=ex["labels"].dtype)
            
            batch_input_ids.append(torch.cat([ex["input_ids"], pad_ids], dim=0))
            batch_attn_mask.append(torch.cat([ex["attention_mask"], pad_mask], dim=0))
            batch_labels.append(torch.cat([ex["labels"], pad_lbls], dim=0))
            if has_mm:
                pad_m = torch.zeros((pad_len,), dtype=ex["mm_token_type_ids"].dtype)
                batch_mm.append(torch.cat([ex["mm_token_type_ids"], pad_m], dim=0))
        else:
            batch_input_ids.append(ex["input_ids"])
            batch_attn_mask.append(ex["attention_mask"])
            batch_labels.append(ex["labels"])
            if has_mm:
                batch_mm.append(ex["mm_token_type_ids"])
                
    out = {
        "input_ids": torch.stack(batch_input_ids, dim=0),
        "attention_mask": torch.stack(batch_attn_mask, dim=0),
        "labels": torch.stack(batch_labels, dim=0),
        "pixel_values": torch.cat([ex["pixel_values"] for ex in batch], dim=0),
        "image_grid_thw": torch.cat([ex["image_grid_thw"].unsqueeze(0) if ex["image_grid_thw"].dim() == 1 else ex["image_grid_thw"] for ex in batch], dim=0)
    }
    if has_mm:
        out["mm_token_type_ids"] = torch.stack(batch_mm, dim=0)
    return out

# Kiểm tra dữ liệu huấn luyện
train_ds = RoadDamageDataset("data/splits/train.jsonl", processor)
sample_item = train_ds[0]
print(f"Tổng số token chuỗi đầy đủ: {sample_item['input_ids'].shape[0]}")
target_tokens = sample_item['input_ids'][sample_item['labels'] != -100]
decoded = processor.tokenizer.decode(target_tokens, skip_special_tokens=False)
print("Giải mã phần token có nhãn labels != -100 (chỉ học phần này):")
print(decoded)


## 5. Setup LoRA / đóng băng cho E1, E2, E3
Cấu hình các thí nghiệm:
- **E1**: LoRA chỉ trên LLM (252 modules: 36 layers × 7 projections `{q,k,v,o}_proj`, `{gate,up,down}_proj`). ViT và Projector đóng băng.
- **E2**: E1 + mở toàn bộ tham số của Projector (`visual.merger` và `visual.deepstack_merger_list`, ~109M tham số) đưa về `float32`.
- **E3**: E2 + LoRA trên các khối attention và MLP của ViT (96 modules: 24 blocks × 4 modules -> tổng 348 modules LoRA).


In [ ]:
from peft import LoraConfig, get_peft_model

def apply_setup(model, setup_name):
    for param in model.parameters():
        param.requires_grad = False
        
    model.enable_input_require_grads()
    
    llm_pattern = r".*language_model\.layers\.\d+\.(self_attn\.(q_proj|k_proj|v_proj|o_proj)|mlp\.(gate_proj|up_proj|down_proj))"
    vit_pattern = r".*visual\.blocks\.\d+\.(attn\.(qkv|proj)|mlp\.(linear_fc1|linear_fc2))"
    
    if setup_name in ("E1", "E2"):
        target_pattern = llm_pattern
    elif setup_name == "E3":
        target_pattern = f"({llm_pattern}|{vit_pattern})"
    else:
        raise ValueError(f"Setup không hợp lệ: {setup_name}")
        
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=target_pattern,
        bias="none",
        task_type="CAUSAL_LM"
    )
    
    peft_model = get_peft_model(model, lora_config)
    matched = [n for n, _ in peft_model.named_modules() if "lora_A" in n]
    print(f"[{setup_name}] Số lượng module gắn LoRA: {len(matched)} (kỳ vọng: 252 cho E1/E2, 348 cho E3)")
    
    if setup_name in ("E2", "E3"):
        proj_params = 0
        for name, param in peft_model.named_parameters():
            if "visual.merger" in name or "visual.deepstack_merger_list" in name:
                param.data = param.data.float()  # Đưa về float32 để ổn định
                param.requires_grad = True
                proj_params += param.numel()
        print(f"[{setup_name}] Tham số Projector được mở (float32): {proj_params:,} (~109M)")
        
    trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in peft_model.parameters())
    print(f"[{setup_name}] Tổng tham số huấn luyện: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
    
    return peft_model


## 6. Vòng huấn luyện
Huấn luyện 300 bước tối ưu với:
- Tối ưu hoá AdamW, lr=2e-4, cosine learning rate scheduler với 5% warmup.
- Batch size = 2, gradient accumulation = 4 (tương đương effective batch size 8).
- Gradient checkpointing (`use_reentrant=False`), `use_cache=False`.
- Lưu adapter LoRA và lưu riêng trọng số `projector.pt` cho E2 và E3.
- Ghi lại nhật ký tài nguyên: `trainable_params`, `peak_vram_gb`, `train_minutes`.


In [ ]:
import os
import sys
import time
import json
import gc
import torch
import subprocess
from tqdm import tqdm
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import get_cosine_schedule_with_warmup

def train_model(setup_name):
    print(f"\n{'='*40}\nBẮT ĐẦU HUẤN LUYỆN: {setup_name}\n{'='*40}")
    
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    gc.collect()
    
    base_m = model_cls.from_pretrained(
        model_id,
        quantization_config=quantization,
        torch_dtype=torch.float16,
        device_map={"": 0}
    )
    base_m.config.use_cache = False
    base_m.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    
    model = apply_setup(base_m, setup_name)
    model.train()
    
    train_loader = DataLoader(
        train_ds,
        batch_size=2,
        shuffle=True,
        collate_fn=lambda b: collate_fn(b, processor.tokenizer.pad_token_id or 0),
        drop_last=True
    )
    
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(trainable_params, lr=2.0e-4, weight_decay=0.0)
    
    max_steps = 300
    grad_accum = 4
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.05 * max_steps),
        num_training_steps=max_steps
    )
    scaler = torch.amp.GradScaler('cuda')
    
    save_dir = f"work/adapters/{setup_name.lower()}"
    os.makedirs(save_dir, exist_ok=True)
    
    start_time = time.time()
    step = 0
    accum_loss = 0.0
    train_iter = iter(train_loader)
    
    pbar = tqdm(total=max_steps, desc=f"Huấn luyện {setup_name}")
    while step < max_steps:
        optimizer.zero_grad()
        for _ in range(grad_accum):
            try:
                batch = next(train_iter)
            except StopIteration:
                train_iter = iter(train_loader)
                batch = next(train_iter)
                
            batch = {k: v.to("cuda") for k, v in batch.items()}
            with torch.amp.autocast('cuda', dtype=torch.float16):
                outputs = model(**batch)
                loss = outputs.loss / grad_accum
                
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Loss bị tràn số {loss.item()} ở bước {step}")
                
            scaler.scale(loss).backward()
            accum_loss += loss.item() * grad_accum
            
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        step += 1
        pbar.update(1)
        pbar.set_postfix({"loss": f"{accum_loss / grad_accum:.4f}", "lr": f"{scheduler.get_last_lr()[0]:.2e}"})
        accum_loss = 0.0
        
        if step % 100 == 0 or step == max_steps:
            model.save_pretrained(save_dir)
            if setup_name in ("E2", "E3"):
                extra = {k: v.cpu() for k, v in model.named_parameters()
                         if ("visual.merger" in k or "visual.deepstack_merger_list" in k) and v.requires_grad}
                torch.save(extra, os.path.join(save_dir, "projector.pt"))
                
    pbar.close()
    
    train_minutes = round((time.time() - start_time) / 60.0, 2)
    peak_vram_gb = round(torch.cuda.max_memory_allocated() / (1024**3), 2)
    num_trainable = sum(p.numel() for p in trainable_params)
    
    run_meta = {
        "setup": setup_name,
        "trainable_params": num_trainable,
        "peak_vram_gb": peak_vram_gb,
        "train_minutes": train_minutes,
        "steps": max_steps
    }
    with open(f"work/{setup_name.lower()}_run.json", "w", encoding="utf-8") as f:
        json.dump(run_meta, f, indent=2)
        
    print(f"[{setup_name}] Hoàn thành: {train_minutes} phút | VRAM đỉnh: {peak_vram_gb} GB | Tham số: {num_trainable:,}")
    
    del model, optimizer, scheduler, scaler, base_m
    torch.cuda.empty_cache()
    gc.collect()
    return run_meta

# Huấn luyện lần lượt E1, E2, E3
for setup in ("E1", "E2", "E3"):
    if not os.path.exists(f"work/{setup.lower()}_run.json"):
        train_model(setup)


## 7. Suy luận và chấm E1–E3
Nạp lại adapter LoRA và trọng số projector đã huấn luyện, thực hiện suy luận trên tập `test` (311 ảnh) và chấm điểm với `scripts/score.py` kèm metadata tài nguyên.


In [ ]:
import os
import sys
import json
import gc
import torch
import subprocess
from peft import PeftModel

def evaluate_setup(setup_name):
    print(f"\n{'='*30}\nĐánh giá {setup_name} trên test set\n{'='*30}")
    torch.cuda.empty_cache()
    gc.collect()
    
    base_m = model_cls.from_pretrained(
        model_id,
        quantization_config=quantization,
        torch_dtype=torch.float16,
        device_map={"": 0}
    )
    
    adapter_dir = f"work/adapters/{setup_name.lower()}"
    peft_m = PeftModel.from_pretrained(base_m, adapter_dir)
    
    proj_path = os.path.join(adapter_dir, "projector.pt")
    if os.path.exists(proj_path):
        print(f"Nạp trọng số projector cho {setup_name}...")
        proj_weights = torch.load(proj_path, map_location="cuda:0")
        peft_m.load_state_dict(proj_weights, strict=False)
        
    peft_m.eval()
    
    out_pred = f"work/{setup_name.lower()}.jsonl"
    run_inference(peft_m, processor, split="test", batch_size=4, out_file=out_pred)
    
    with open(f"work/{setup_name.lower()}_run.json", "r") as f:
        meta_info = json.load(f)
        
    out_evidence = f"report/evidence/{setup_name.lower()}.json"
    subprocess.run([
        sys.executable, "scripts/score.py",
        "--pred", out_pred,
        "--name", setup_name,
        "--out", out_evidence,
        f"--meta=trainable_params={meta_info['trainable_params']}",
        f"--meta=peak_vram_gb={meta_info['peak_vram_gb']}",
        f"--meta=train_minutes={meta_info['train_minutes']}"
    ], check=True)
        
    del peft_m, base_m
    torch.cuda.empty_cache()
    gc.collect()

for setup in ("E1", "E2", "E3"):
    evaluate_setup(setup)


## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)
- **E4 Ablation**: Chọn biến LoRA rank `r=4` so với `r=16` và `r=64` (đặt alpha = 2*r) trên setup tốt nhất (ví dụ E2).
- **E5 (Bonus)**: Cài đặt và kiểm thử hàm reward IoU cho Grounding RL (GRPO).


In [ ]:
# E4: Ablation theo Rank LoRA (r=4 và r=64)
import os
import sys
import json
import time
import gc
import torch
import subprocess
from tqdm import tqdm
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model

def run_e4_rank(rank=4, base_setup="E2"):
    setup_name = f"E4_rank_{rank}"
    print(f"\n{'='*30}\nChạy Ablation: {setup_name}\n{'='*30}")
    
    torch.cuda.empty_cache()
    gc.collect()
    
    base_m = model_cls.from_pretrained(
        model_id,
        quantization_config=quantization,
        torch_dtype=torch.float16,
        device_map={"": 0}
    )
    base_m.config.use_cache = False
    base_m.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    
    for p in base_m.parameters():
        p.requires_grad = False
    base_m.enable_input_require_grads()
    
    target_pattern = r".*language_model\.layers\.\d+\.(self_attn\.(q_proj|k_proj|v_proj|o_proj)|mlp\.(gate_proj|up_proj|down_proj))"
    lora_cfg = LoraConfig(r=rank, lora_alpha=2*rank, lora_dropout=0.05, target_modules=target_pattern, bias="none", task_type="CAUSAL_LM")
    peft_m = get_peft_model(base_m, lora_cfg)
    
    for name, param in peft_m.named_parameters():
        if "visual.merger" in name or "visual.deepstack_merger_list" in name:
            param.data = param.data.float()
            param.requires_grad = True
            
    peft_m.train()
    
    train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, collate_fn=lambda b: collate_fn(b, processor.tokenizer.pad_token_id or 0), drop_last=True)
    trainable_params = [p for p in peft_m.parameters() if p.requires_grad]
    optimizer = AdamW(trainable_params, lr=2.0e-4, weight_decay=0.0)
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=15, num_training_steps=300)
    scaler = torch.amp.GradScaler('cuda')
    
    save_dir = f"work/adapters/{setup_name.lower()}"
    os.makedirs(save_dir, exist_ok=True)
    
    start_t = time.time()
    train_iter = iter(train_loader)
    for step in tqdm(range(300), desc=f"Train {setup_name}"):
        optimizer.zero_grad()
        for _ in range(4):
            try:
                batch = next(train_iter)
            except StopIteration:
                train_iter = iter(train_loader)
                batch = next(train_iter)
            batch = {k: v.to("cuda") for k, v in batch.items()}
            with torch.amp.autocast('cuda', dtype=torch.float16):
                loss = peft_m(**batch).loss / 4
            scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
    peft_m.save_pretrained(save_dir)
    extra = {k: v.cpu() for k, v in peft_m.named_parameters() if ("visual.merger" in k or "visual.deepstack_merger_list" in k) and v.requires_grad}
    torch.save(extra, os.path.join(save_dir, "projector.pt"))
    
    t_min = round((time.time() - start_t) / 60.0, 2)
    vram = round(torch.cuda.max_memory_allocated() / (1024**3), 2)
    n_params = sum(p.numel() for p in trainable_params)
    
    # Suy luận & chấm
    peft_m.eval()
    pred_file = f"work/{setup_name.lower()}.jsonl"
    run_inference(peft_m, processor, split="test", batch_size=4, out_file=pred_file)
    evidence_file = f"report/evidence/e4_rank_{rank}.json"
    subprocess.run([
        sys.executable, "scripts/score.py",
        "--pred", pred_file,
        "--name", setup_name,
        "--out", evidence_file,
        f"--meta=trainable_params={n_params}",
        f"--meta=peak_vram_gb={vram}",
        f"--meta=train_minutes={t_min}"
    ], check=True)
        
    del peft_m, base_m
    torch.cuda.empty_cache()
    gc.collect()

# Chạy E4 với rank=4
if not os.path.exists("report/evidence/e4_rank_4.json"):
    run_e4_rank(rank=4)


In [ ]:
# E5: Kiểm thử Hàm reward IoU cho GRPO (RL)
from scripts.score import parse_output, iou_xyxy, CLASSES

def compute_grounding_reward(completion, gt_boxes):
    """    Hàm tính reward kiểm chứng được cho Grounding:
    - completion: chuỗi text output của model
    - gt_boxes: danh sách box ground truth [{"label": ..., "bbox": (x1, y1, x2, y2)}]
    """
    pred = parse_output(completion)
    if pred is None:
        return 0.0  # Phạt nặng JSON không hợp lệ
    if not gt_boxes and not pred:
        return 1.0  # Cả 2 đều rỗng (ảnh không hư hại)
    if not gt_boxes or not pred:
        return 0.2  # JSON hợp lệ nhưng dự đoán sai hoàn toàn
        
    matched_ious = []
    for lab in CLASSES:
        g = [x["bbox"] for x in gt_boxes if x["label"] == lab]
        p = [x["bbox"] for x in pred if x["label"] == lab]
        pairs = sorted(((iou_xyxy(a, b), i, j) for i, a in enumerate(g) for j, b in enumerate(p)), reverse=True)
        used_g, used_p = set(), set()
        for iou, i, j in pairs:
            if iou < 0.5:
                break
            if i in used_g or j in used_p:
                continue
            used_g.add(i)
            used_p.add(j)
            matched_ious.append(iou)
            
    total_iou = sum(matched_ious)
    denom = max(len(gt_boxes), len(pred), 1)
    return float(0.2 + 0.8 * (total_iou / denom))

# Unit test hàm reward
gt_test = [{"label": "pothole", "bbox": (100, 100, 300, 300)}]
assert compute_grounding_reward('[{"label": "pothole", "bbox_2d": [100, 100, 300, 300]}]', gt_test) == 1.0
assert compute_grounding_reward('invalid output', gt_test) == 0.0
assert compute_grounding_reward('[{"label": "crack", "bbox_2d": [100, 100, 300, 300]}]', gt_test) == 0.2
r_extra = compute_grounding_reward('[{"label": "pothole", "bbox_2d": [100, 100, 300, 300]}, {"label": "pothole", "bbox_2d": [500, 500, 600, 600]}]', gt_test)
assert 0.2 < r_extra < 1.0
print("Unit test hàm reward IoU: TẤT CẢ TEST ĐỀU PASS!")


In [ ]:
# Sinh ảnh so sánh Ground Truth vs Prediction cho báo cáo
import matplotlib.pyplot as plt

def generate_comparison_images(pred_jsonl, split="test", out_path="report/images/pred_comparison.png"):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    preds = {r["image_id"]: parse_output(r["raw"]) for r in load_jsonl(pred_jsonl)}
    test_entries = load_jsonl(f"data/splits/{split}.jsonl")
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    for idx in range(min(6, len(test_entries))):
        e = test_entries[idx]
        img = Image.open(f"data/{e['image_path']}")
        w, h = e["width"], e["height"]
        ax = axes[idx]
        ax.imshow(img)
        ax.set_title(f"Test: {e['image_id']}", fontsize=10)
        ax.axis("off")
        
        # Ground Truth: Màu xanh lá cây (solid)
        for o in e["objects"]:
            xc, yc, bw, bh = o["yolo"]
            x1, y1 = (xc - bw/2)*w, (yc - bh/2)*h
            rect = patches.Rectangle((x1, y1), bw*w, bh*h, linewidth=2, edgecolor="lime", facecolor="none")
            ax.add_patch(rect)
            ax.text(x1, max(0, y1-3), f"GT:{o['label']}", color="lime", fontsize=8,
                    bbox=dict(boxstyle="square,pad=0.1", fc="black", ec="lime", alpha=0.7))
                    
        # Prediction: Màu đỏ / tím (dashed)
        pred_boxes = preds.get(e["image_id"]) or []
        for p in pred_boxes:
            px1, py1 = p["bbox"][0]/1000*w, p["bbox"][1]/1000*h
            px2, py2 = p["bbox"][2]/1000*w, p["bbox"][3]/1000*h
            rect = patches.Rectangle((px1, py1), px2-px1, py2-py1, linewidth=2, edgecolor="magenta", linestyle="--", facecolor="none")
            ax.add_patch(rect)
            ax.text(px1, min(h, py2+10), f"Pred:{p['label']}", color="magenta", fontsize=8,
                    bbox=dict(boxstyle="square,pad=0.1", fc="black", ec="magenta", alpha=0.7))
                    
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.show()
    print(f"Đã lưu ảnh so sánh vào: {out_path}")

if os.path.exists("work/e2.jsonl"):
    generate_comparison_images("work/e2.jsonl")
elif os.path.exists("work/e1.jsonl"):
    generate_comparison_images("work/e1.jsonl")


In [ ]:
# Kiểm tra bài nộp và sao chép kết quả về output của Kaggle
import sys
import subprocess
subprocess.run([sys.executable, "scripts/check_submission.py"], check=False)

if IN_KAGGLE and os.path.abspath(os.getcwd()) != os.path.abspath("/kaggle/working"):
    import shutil
    print("Sao chép kết quả về /kaggle/working/...")
    if os.path.exists("report"):
        shutil.copytree("report", "/kaggle/working/report", dirs_exist_ok=True)
    if os.path.exists("work"):
        os.makedirs("/kaggle/working/work", exist_ok=True)
        for item in os.listdir("work"):
            src_item = os.path.join("work", item)
            if os.path.isfile(src_item):
                shutil.copy2(src_item, os.path.join("/kaggle/working/work", item))
    print("Đã sao chép toàn bộ kết quả ra /kaggle/working/ để tải về!")
